# Reactor Point Kinetics

Simulating the transient neutron population in a nuclear reactor using the point kinetics equations (PKE) with six delayed neutron precursor groups.

The point kinetics model couples the neutron density $n$ with $G$ delayed neutron precursor groups $C_i$:

$$\frac{dn}{dt} = \frac{\rho - \beta}{\Lambda} \, n + \sum_{i=1}^{G} \lambda_i \, C_i + S$$

$$\frac{dC_i}{dt} = \frac{\beta_i}{\Lambda} \, n - \lambda_i \, C_i \qquad i = 1, \ldots, G$$

where $\beta = \sum_i \beta_i$ is the total delayed neutron fraction, $\Lambda$ is the prompt neutron generation time, and $\rho$ is the reactivity.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('../pathsim_docs.mplstyle')

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope
from pathsim.solvers import GEAR52A

from pathsim_chem.neutronics import PointKinetics

The point kinetics system is very stiff — the prompt neutron generation time $\Lambda \sim 10^{-5}\,\text{s}$ creates eigenvalues on the order of $10^5$. The variable-order BDF solver GEAR52A is ideal here: it requires only one implicit solve per step and adapts both step size and order to the smooth exponential dynamics.

## 1. Delayed Supercritical Step

Insert a step reactivity of $\rho = 0.003$ (about $0.46\beta$). Since $\rho < \beta$, the reactor is delayed supercritical — the power rises on a slow time scale governed by the delayed neutrons.

In [ ]:
reactor = PointKinetics(n0=1.0)

rho_step = 0.003
src_rho = Source(lambda t: rho_step if t > 0 else 0.0)
src_s = Source(lambda t: 0.0)
sco = Scope(labels=["n"])

sim = Simulation(
    [src_rho, src_s, reactor, sco],
    [
        Connection(src_rho, reactor),       # rho
        Connection(src_s, reactor[1]),       # S
        Connection(reactor, sco),            # n
    ],
    dt=0.01,
    Solver=GEAR52A,
    log=True,
)

sim.run(100)

In [ ]:
sco.plot(lw=1.5)
plt.yscale('log')
plt.title('Delayed Supercritical (ρ = 0.003)')
plt.show()

The neutron density rises exponentially on a time scale of seconds — much slower than the prompt neutron lifetime ($\Lambda \sim 10^{-5}$ s) because the delayed neutrons control the dynamics when $\rho < \beta$.

## 2. Prompt Supercritical

Insert $\rho = 0.008 > \beta \approx 0.0065$. Now the reactor is prompt supercritical — the power rises on the prompt neutron time scale, producing a rapid excursion.

In [ ]:
reactor = PointKinetics(n0=1.0)

rho_prompt = 0.008
src_rho = Source(lambda t: rho_prompt if t > 0 else 0.0)
src_s = Source(lambda t: 0.0)
sco = Scope(labels=["n"])

sim = Simulation(
    [src_rho, src_s, reactor, sco],
    [
        Connection(src_rho, reactor),
        Connection(src_s, reactor[1]),
        Connection(reactor, sco),
    ],
    dt=0.001,
    Solver=GEAR52A,
    log=True,
)

sim.run(0.5)

In [ ]:
sco.plot(lw=1.5)
plt.yscale('log')
plt.title('Prompt Supercritical (ρ = 0.008 > β)')
plt.show()

The power rises orders of magnitude within milliseconds. This is why prompt criticality must be avoided in reactor design — the delayed neutrons are the key safety mechanism that keeps power transients manageable.

## 3. Subcritical with External Source

A subcritical assembly ($\rho = -0.05$) with a constant external neutron source. The system reaches an equilibrium where the source multiplication produces a steady neutron population:

$$n_{ss} = \frac{-S \, \Lambda}{\rho}$$

In [ ]:
rho_sub = -0.05
s_ext = 1e5
Lambda = 1e-5

reactor = PointKinetics(n0=0.001, Lambda=Lambda)

src_rho = Source(lambda t: rho_sub)
src_s = Source(lambda t: s_ext)
sco = Scope(labels=["n"])

sim = Simulation(
    [src_rho, src_s, reactor, sco],
    [
        Connection(src_rho, reactor),
        Connection(src_s, reactor[1]),
        Connection(reactor, sco),
    ],
    dt=0.01,
    Solver=GEAR52A,
    log=True,
)

sim.run(50)

In [ ]:
n_ss = -s_ext * Lambda / rho_sub

sco.plot(lw=1.5)
plt.axhline(n_ss, color='r', ls='--', lw=1, label=f'$n_{{ss}}$ = {n_ss:.4f}')
plt.legend()
plt.title('Subcritical with Source (ρ = −0.05)')
plt.show()

The neutron density converges to the expected source-multiplied equilibrium value, confirming the subcritical multiplication physics.